In [ ]:
import sys
sys.path.append('..')
sys.path.append('../../wildlife-datasets')
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from wildlife_datasets.datasets import utils
from animal_re_id.turtlewatch_egypt import TWE_NewData, TWE_Master, fix_chars, fix_identity, merge_datasets

In [ ]:
root_master = '/data/wildlife_datasets/turtle_watch/TWE_flipperprofiles_registered ind_LR'
root_new = '/data/wildlife_datasets/turtle_watch'
roots_new = [
    '2017 PICS + DATASHEET',
    '2018 PICS + DATASHEET',
    '2019 PICS + DATASHEET',
    '2020 PICS + DATASHEET',
    '2021 PICS + DATASHEET',
    '2022 PICS + online database',
    '2023 PICS + online database',
    '2024 PICS + online database',
    '2025 PICS + online database',    
]
individuals_all = TWE_NewData.individuals

dataset = merge_datasets(root_new, roots_new, img_load='bbox')
np.setdiff1d(dataset.df['identity'], individuals_all)

In [ ]:
root = '/data/wildlife_datasets/turtle_watch/PROFILES'
# root = '/data/wildlife_datasets/turtle_watch/TWE_flipperprofiles_registered ind_LR'

TWE_Master.get_data(root, force=True)
dataset = TWE_Master(root)

In [ ]:
len(dataset.identities)

In [ ]:
np.setdiff1d(dataset.identities, individuals_all)

In [ ]:
np.setdiff1d(individuals_all, dataset.identities)

In [ ]:
root = '/data/wildlife_datasets/turtle_watch/PROFILES_SITES'

data = utils.find_images(root)
idx = data['path'].apply(lambda x: 'noid' in x.lower())
data = data[~idx]
idx = data['path'].apply(lambda x: 'no id' in x.lower())
data = data[~idx]
folders = data['path'].str.split(os.path.sep, expand=True)

identities = []
n_cols = np.max(folders.columns) + 1
for _, df_row in folders.iterrows():
    asd = []
    for i in range(2, n_cols):
        if pd.isnull(df_row[i]):
            continue
        zxc = fix_identity(fix_chars(df_row[i]), individuals_all)
        if zxc in individuals_all:
            asd.append(zxc)
            break
    if len(asd) == 1:
        identities.append(asd[0])
    else:
        identities.append('unknown')
identities = np.array(identities)

np.setdiff1d(individuals_all, identities)